# 01  Data Collection

This notebook downloads and inspects all raw data used by the model:
- **ETF prices** — IVV.AX, VAS.AX, NDQ.AX, VGS.AX
- **Macro indicators** — VIX, 10Y Treasury yield, Gold, S&P 500, AUD/USD

All data is sourced from Yahoo Finance via `yfinance`. Nothing is transformed here — that happens in notebook 02.

## 1. Imports

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

from data_loader import (
    TICKERS, MACRO_TICKERS,
    TRAIN_START, TRAIN_END,
    TEST_START, TEST_END,
    download_data, load_macro_data
)

print('All imports successful')
print(f'ETFs     : {TICKERS}')
print(f'Macro    : {MACRO_TICKERS}')
print(f'Training : {TRAIN_START} → {TRAIN_END}')
print(f'Test     : {TEST_START} → {TEST_END}')

## 2. Download ETF Training Data

Training window: **January 2022 – December 2024** (post-COVID market conditions).

In [ ]:
etf_data = {}

for ticker in TICKERS:
    print(f'Downloading {ticker}...')
    df = download_data(ticker, TRAIN_START, TRAIN_END)
    # Flatten MultiIndex columns if present (yfinance >= 0.2)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    etf_data[ticker] = df
    print(f'  {ticker}: {len(df)} trading days, {df.shape[1]} columns')

print('\nDone.')

## 3. Inspect a Sample ETF

Let's look at IVV.AX as a representative example.

In [ ]:
sample = etf_data['IVV.AX']

print('=== IVV.AX — First 5 rows ===')
display(sample.head())

print('\n=== IVV.AX — Last 5 rows ===')
display(sample.tail())

print('\n=== Data types ===')
print(sample.dtypes)

print('\n=== Missing values ===')
print(sample.isnull().sum())

## 4. Summary Statistics: All ETFs

In [ ]:
summary_rows = []

for ticker, df in etf_data.items():
    summary_rows.append({
        'Ticker'     : ticker,
        'Trading days': len(df),
        'Start'      : df.index[0].strftime('%Y-%m-%d'),
        'End'        : df.index[-1].strftime('%Y-%m-%d'),
        'Min Close'  : f"${df['Close'].min():.2f}",
        'Max Close'  : f"${df['Close'].max():.2f}",
        'Avg Volume' : f"{df['Volume'].mean():,.0f}",
        'Missing'    : df.isnull().sum().sum()
    })

summary_df = pd.DataFrame(summary_rows).set_index('Ticker')
display(summary_df)

## 5. Plot Raw Closing Prices: All ETFs

A first look at the price history across the training window.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for i, (ticker, df) in enumerate(etf_data.items()):
    ax = axes[i]
    ax.plot(df.index, df['Close'], color=colors[i], linewidth=1.2)
    ax.set_title(f'{ticker} — Closing Price (Training)', fontsize=12)
    ax.set_ylabel('Price (AUD)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Raw Closing Prices — Training Data (2022–2024)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Download Macro Indicators

In [ ]:
print('Downloading macro indicators...')
macro_train = load_macro_data(TRAIN_START, TRAIN_END)

print(f'\nShape: {macro_train.shape}')
print(f'Date range: {macro_train.index[0].strftime("%Y-%m-%d")} → {macro_train.index[-1].strftime("%Y-%m-%d")}')
print(f'\nMissing values per column:')
print(macro_train.isnull().sum())
display(macro_train.head())

## 7. Plot Macro Indicators

In [ ]:
macro_labels = {
    '^VIX'    : ('VIX — Market Fear Index',      '#E53935'),
    '^TNX'    : ('10Y Treasury Yield (%)',        '#FB8C00'),
    'GLD'     : ('Gold ETF Price (USD)',          '#FDD835'),
    '^GSPC'   : ('S&P 500 Index',                 '#43A047'),
    'AUDUSD=X': ('AUD/USD Exchange Rate',         '#1E88E5'),
}

fig, axes = plt.subplots(5, 1, figsize=(14, 14))

for i, (col, (label, color)) in enumerate(macro_labels.items()):
    if col in macro_train.columns:
        axes[i].plot(macro_train.index, macro_train[col], color=color, linewidth=1.2)
        axes[i].set_title(label, fontsize=11)
        axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        axes[i].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
        plt.setp(axes[i].xaxis.get_majorticklabels(), rotation=30, ha='right')
        axes[i].grid(True, alpha=0.3)

plt.suptitle('Macro Indicators — Training Window (2022–2024)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 8. Check Date Alignment

ETF and macro data must share overlapping trading dates before joining in the feature engineering step.

In [ ]:
print('=== Date alignment check ===')
for ticker, df in etf_data.items():
    common = df.index.intersection(macro_train.index)
    print(f'{ticker}: {len(df)} ETF days | {len(macro_train)} macro days | {len(common)} common dates')

print('\nAll ETFs and macro data align on shared trading dates.')
print('Forward-fill (ffill) in features.py handles any remaining gaps.')

## 9. Download Test Data (2025)

Quick sanity check that 2025 data is available before training.

In [ ]:
print('Checking 2025 test data availability...\n')

for ticker in TICKERS:
    df_test = download_data(ticker, TEST_START, TEST_END)
    if isinstance(df_test.columns, pd.MultiIndex):
        df_test.columns = df_test.columns.get_level_values(0)
    print(f'{ticker}: {len(df_test)} trading days available in 2025')
    print(f'  From {df_test.index[0].strftime("%Y-%m-%d")} to {df_test.index[-1].strftime("%Y-%m-%d")}')
    print(f'  Close range: ${df_test["Close"].min():.2f} – ${df_test["Close"].max():.2f}\n')

## 10. Save Raw Data to CSV

Saves downloads to `data/raw/` so you don't need to re-download on every run.

In [ ]:
os.makedirs('data/raw', exist_ok=True)

for ticker, df in etf_data.items():
    filename = f"data/raw/{ticker.replace('.', '_')}_train.csv"
    df.to_csv(filename)
    print(f'Saved {filename}')

macro_train.to_csv('data/raw/macro_train.csv')
print('Saved data/raw/macro_train.csv')

print('\nAll raw data saved to data/raw/')

## Summary

| | |
|---|---|
| ETFs downloaded | IVV.AX, VAS.AX, NDQ.AX, VGS.AX |
| Training window | Jan 2022 – Dec 2024 |
| Macro indicators | VIX, TNX, GLD, GSPC, AUDUSD |
| Test data | 2025 confirmed available |
| Raw data saved | `data/raw/` |

**Next → `02_eda_and_preprocessing.ipynb`** — feature engineering and exploratory analysis.